In [2]:
from scipy.stats import binomtest
from pathlib import Path
import sys
import numpy as np

_candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
PROJECT_ROOT = next(p for p in _candidates if (p / "src" / "constellation_denoiser").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from constellation_denoiser import *

In [3]:
tx, rx = load_channel_data("Symbols_1m_1ch_PR_1.csv")
(tx_train, rx_train), (tx_val, rx_val), (tx_test, rx_test) = split(tx, rx)

## Per-class bias (fit on train)

In [4]:
tx_idx_train = get_indexes(tx_train)
noise_train = rx_train - tx_train

bias = np.zeros(16, dtype=complex)
for i in range(16):
    bias[i] = noise_train[tx_idx_train == i].mean()

## Baseline vs decision-directed correction (val, matched split)

In [6]:
tx_idx_val = get_indexes(tx_val)
rx_idx_val = get_indexes(rx_val)
ber_raw = calculate_ber(BIT_TABLE[rx_idx_val].reshape(-1), BIT_TABLE[tx_idx_val].reshape(-1))
print(f"BER raw = {ber_raw:.6f}")

corrected = rx_val - bias[rx_idx_val]
corrected_idx = get_indexes(corrected)
corrected = rx_val - bias[corrected_idx]
corrected_idx = get_indexes(corrected)
ber_corr = calculate_ber(BIT_TABLE[corrected_idx].reshape(-1), BIT_TABLE[tx_idx_val].reshape(-1))
print(f"BER corrected = {ber_corr:.6f}")

BER raw = 0.008747
BER corrected = 0.008442


## Oracle bound (uses true labels, not achievable — diagnostic only)

In [7]:
noise_val = rx_val - tx_val
bias_val = np.zeros(16, dtype=complex)
for i in range(16):
    bias_val[i] = noise_val[tx_idx_val == i].mean()

oracle_idx = get_indexes(rx_val - bias_val[tx_idx_val])
ber_oracle = calculate_ber(BIT_TABLE[oracle_idx].reshape(-1), BIT_TABLE[tx_idx_val].reshape(-1))
print(f"BER oracle = {ber_oracle:.6f}")

BER oracle = 0.007832


## Is the improvement statistically real?

In [8]:
n_bits = len(tx_idx_val) * 4
se = np.sqrt(0.008 * 0.992 / n_bits)
print(f"standard error of BER estimate = {se:.5f}")

bits_true = BIT_TABLE[tx_idx_val].reshape(-1)
bits_raw = BIT_TABLE[rx_idx_val].reshape(-1)
bits_corr = BIT_TABLE[corrected_idx].reshape(-1)

raw_wrong_corr_right = np.sum((bits_raw != bits_true) & (bits_corr == bits_true))
raw_right_corr_wrong = np.sum((bits_raw == bits_true) & (bits_corr != bits_true))
p = binomtest(min(raw_wrong_corr_right, raw_right_corr_wrong), raw_wrong_corr_right + raw_right_corr_wrong, 0.5).pvalue
print(f"McNemar p-value = {p:.4f}")

standard error of BER estimate = 0.00090
McNemar p-value = 0.4531
